# 04 · Cleanup: what you said becomes what you meant

One LLM call. System prompt = the rules, user message = the transcript.

## 4.1 · Setup, and a messy transcript

Written by hand, one of each problem planted: fillers, a false start, "Tuesday, sorry, Wednesday", no punctuation.

In [ ]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from groq import Groq

load_dotenv(Path("../.env"))
client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL = "qwen/qwen3.8-27b"

MESSY = (
    "um so I was thinking we should probably move the meeting to Tuesday "
    "sorry Wednesday because uh John is out and also like can you send me the "
    "can you send me the updated deck before that not the old one you know thanks"
)

print(f"model : {MODEL}")
print(f"input : {len(MESSY.split())} words\n")
print(MESSY)

## 4.2 · The naive prompt

```python
"Clean up this voice transcription and make it sound good:"
```

In [6]:
def clean(prompt: str, transcript: str, model: str = MODEL) -> tuple[float, int, str]:
    """Run one cleanup pass. Returns (latency_ms, output_tokens, text)."""
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": transcript},
        ],
        temperature=0,
        reasoning_effort="none",   # as in 08: no thinking, keeps the ~200ms budget
    )
    ms = (time.perf_counter() - t0) * 1000
    return ms, r.usage.completion_tokens, r.choices[0].message.content


NAIVE = "Clean up this voice transcription and make it sound good. Output only the cleaned text:"
ms, tokens, out = clean(NAIVE, MESSY)
print(f"{ms:.0f} ms   {tokens} output tokens\n")
print(out)

354 ms   38 output tokens

I was thinking we should probably move the meeting to Wednesday, as John is out. Also, could you please send me the updated deck before then? Not the old one. Thanks.


## 4.3 · The strict prompt

A whitelist: remove fillers, resolve corrections, punctuate, add nothing.

In [8]:
STRICT = """
You clean up voice dictation transcripts.

Rules:
- Output ONLY the cleaned text. No preamble, no quotes, no explanation, no sign-off.
- Remove filler words: um, uh, like, you know, I mean, so at the start of a sentence.
- Resolve false starts and self-corrections: keep only what the speaker settled on,
  and delete the word they corrected away from.
- Add punctuation, capitalisation and sentence breaks.
"""

naive_ms, naive_tok, naive_out = clean(NAIVE, MESSY)
strict_ms, strict_tok, strict_out = clean(STRICT, MESSY)

for label, ms, tok, out in [
    ("NAIVE", naive_ms, naive_tok, naive_out),
    ("STRICT", strict_ms, strict_tok, strict_out),
]:
    print(f"=== {label:6s}  {ms:6.0f} ms   {tok:3d} output tokens ===")
    print(out)
    print()

print(f"tokens : {naive_tok} -> {strict_tok}   ({100*(1-strict_tok/naive_tok):.0f}% fewer)")
print(f"latency: {naive_ms:.0f} ms -> {strict_ms:.0f} ms   ({100*(1-strict_ms/naive_ms):.0f}% faster)")

=== NAIVE      417 ms    38 output tokens ===
I was thinking we should probably move the meeting to Wednesday, as John is out. Also, could you please send me the updated deck before then? Not the old one. Thanks.

=== STRICT     499 ms    36 output tokens ===
I was thinking we should probably move the meeting to Wednesday because John is out. Also, can you send me the updated deck before that, not the old one? Thanks.

tokens : 38 -> 36   (5% fewer)
latency: 417 ms -> 499 ms   (-20% faster)


## 4.4 · Context: where the text is going

Same transcript, same prompt, plus one line: Slack or Gmail.

In [9]:
APPS = {
    "Slack":    "The text is going into Slack and should be relaxed",
    "Gmail":    "The text is going into Gmail and should be professional"
}

print(f"{'baseline':10s} {strict_ms:5.0f} ms  {strict_tok:3d} tok")
print(f"{'':10s} {strict_out}\n")

for app, instruction in APPS.items():
    ms, tok, out = clean(STRICT + "\n\nDestination context: " + instruction +
      ". Rewrite the wording and tone to fit, and you may add greetings or politeness.", MESSY)

    print(f"{app:10s} {ms:5.0f} ms  {tok:3d} tok")
    print(f"{'':10s} {out}\n")

baseline     499 ms   36 tok
           I was thinking we should probably move the meeting to Wednesday because John is out. Also, can you send me the updated deck before that, not the old one? Thanks.

Slack        272 ms   39 tok
           Hey team! Quick update: let's move the meeting to Wednesday since John is out. Also, could you send over the updated deck (not the old one) before then? Thanks!

Gmail        324 ms   54 tok
           Hi,

Could we please move the meeting to Wednesday? John is unavailable on Tuesday.

Also, could you send me the updated deck before then? Please make sure it’s the latest version, not the old one.

Thanks,
[Your Name]

